# SVD

Разложение на три матрицы позволяет сжимать данные и при этом терять минимум информации

Пример реализации на сжатии картинки:

In [6]:
import numpy as np
from PIL import Image

image = Image.open('The_Jackal.jpg').convert('L')
image_matrix = np.array(image, dtype=float)

# Создание списка главных компонент
max_k = min(image_matrix.shape)  # максимальное значение которое можно задать, это минимальная ось матрицы
list_k = [1]
while True:
    new_value = list_k[-1] * 2
    if new_value < max_k:
        list_k.append(new_value)
    else:
        list_k.append(max_k)
        break
print(list_k)

U, S, Vt = np.linalg.svd(image_matrix, full_matrices=False)
for k in list_k:
    compressed_matrix = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    compressed_matrix = np.clip(compressed_matrix, 0, 255).astype(np.uint8)
    compressed_image = Image.fromarray(compressed_matrix, mode='L')
    compressed_image.save(f'Compressed_image_k_{k}.jpg')

[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1000]


1 главная компонента:  
![Картинка](Compressed_image_k_1.jpg)  
2 главной компоненты:  
![Картинка](Compressed_image_k_2.jpg)  
4 главной компоненты:  
![Картинка](Compressed_image_k_4.jpg)  
8 главных компонент:  
![Картинка](Compressed_image_k_8.jpg)  
16 главных компонент:  
![Картинка](Compressed_image_k_16.jpg)  
32 главные компоненты:  
![Картинка](Compressed_image_k_32.jpg)  
64 главных компонент:  
![Картинка](Compressed_image_k_64.jpg)  
128 главных компонент:  
![Картинка](Compressed_image_k_128.jpg)  
256 главных компонент:  
![Картинка](Compressed_image_k_256.jpg)  
512 главных компонент:  
![Картинка](Compressed_image_k_512.jpg)  
Изначальная картинка (1000 компонент):  
![Картинка](Compressed_image_k_1000.jpg)  

Такую же реализацию можно сделать через следующий код:  

In [ ]:
import numpy as np
from PIL import Image
from sklearn.decomposition import TruncatedSVD

# Загружаем и преобразуем в матрицу
image = Image.open('The_Jackal.jpg').convert('L')
image_matrix = np.array(image, dtype=float)

# Создание списка главных компонент
max_k = min(image_matrix.shape)  # максимальное значение которое можно задать, это минимальная ось матрицы
list_k = [1]
while True:
    new_value = list_k[-1] * 2
    if new_value < max_k:
        list_k.append(new_value)
    else:
        list_k.append(max_k)
        break

for k in list_k:
    # Создаём модель, говорим оставить k компонент
    svd = TruncatedSVD(n_components=k, random_state=42)
    
    # Обучаем и преобразуем (получаем U_k * S_k размером (M, k))
    reduced = svd.fit_transform(image_matrix)
    
    # Восстанавливаем приближение (M, N)
    reconstructed = svd.inverse_transform(reduced)
    
    # Приводим к формату изображения (uint8, диапазон 0–255)
    compressed = np.clip(reconstructed, 0, 255).astype(np.uint8)
    compressed_image = Image.fromarray(compressed, mode='L')
    compressed_image.save(f'image_k_{k}.jpg')